← [Redes neuronales](06-redes-neuronales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cambio de dominio](08-cambio-de-dominio.ipynb) →

# 07 · Transfer learning y MobileNet



## Estado vigente del proyecto (actualizado el 13 de agosto de 2026)

Esta serie conserva explicaciones y resultados históricos, pero la referencia
operativa actual es la siguiente:

- El sistema experto tiene **193 reglas**, CF estilo MYCIN, meta-reglas,
  encadenamiento hacia adelante y hacia atrás. Su voto usa OpenAI como
  proveedor principal, con heurísticas OpenCV que refinan atributos.
- El modelo local que participa en la decisión es **MobileNetV2 TFLite
  float32**, corrida `run_20260721_2129`; clasifica solo `plastico | vidrio`.
  MobileNetV3-Large INT8 está archivado como respaldo y **no emite votos**.
- En 1.000 capturas OV3660/QVGA, V2 obtuvo **71,60 %** de exactitud y
  **71,25 %** de macro-F1; V3 INT8 obtuvo 57,10 % y 57,09 %. La validación
  histórica de 98,43 % no describe por sí sola el rendimiento del robot.
- La ESP32-CAM toma tres fotos: se suman los seis votos válidos de ambas
  fuentes. `desconocido` es abstención; un empate se resuelve con el proveedor.
  Si el proveedor se abstiene las tres veces, el modelo local necesita 3/3.

La documentación operativa es [`ia/vision-service/README.md`](../../ia/vision-service/README.md),
[`model/README.md`](../../ia/vision-service/model/README.md) y
[`PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md`](../../docs/PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md).


## El problema del tamaño

Entrenar una CNN desde cero necesita millones de imágenes. Reci reunió 18.029 en total y usó 17.630 para entrenamiento.
Con tan pocas, una red grande **memoriza** el conjunto de entrenamiento en vez
de aprender a distinguir materiales: acierta todo lo que ya vio y falla en lo
nuevo. Eso se llama **sobreajuste** (*overfitting*).


## Transfer learning

La solución aprovecha algo del documento anterior: las primeras capas de
cualquier CNN aprenden bordes, texturas y brillos. **Eso es igual para
cualquier tarea visual** — un detector de bordes sirve tanto para reconocer
perros como botellas.

Entonces:

```
1. tomar una red ya entrenada con millones de fotos genéricas (ImageNet)
2. congelar sus capas: ya saben ver
3. reemplazar solo la última capa por una nueva de 2 salidas
4. entrenar esa capa con las fotos propias
```

La red no aprende a ver desde cero: aprende a **traducir** lo que ya sabe ver al
vocabulario del problema. Por eso 26.000 fotos alcanzan.

**ImageNet** es el dataset de referencia: ~1,2 millones de fotos en 1.000
categorías. Ninguna es "plástico" o "vidrio", y no importa — lo que se hereda
es la capacidad de extraer características visuales.



## Las dos fases

En Reci el entrenamiento tiene dos etapas, visibles en `entrenador.py`:

| Fase | Qué se entrena | Tasa de aprendizaje | Resultado histórico |
| --- | --- | --- | --- |
| 1 · Cabeza | Solo la capa final | `1e-3` (alta) | 90,12 % |
| 2 · Ajuste fino | Últimas capas del backbone + cabeza | `1e-5` (100× menor) | 98,43 % |

La lógica del orden: si se descongelara el backbone desde el principio, la
capa final —que empieza con valores aleatorios— produciría errores enormes que
destruirían los pesos buenos de ImageNet. Primero se estabiliza la cabeza,
después se afina el resto **con pasos muy pequeños** para no borrar lo heredado.

Un detalle del código que responde a lo mismo: las capas de `BatchNormalization`
se mantienen congeladas durante la fase 2. Con lotes pequeños, actualizar sus
estadísticas desestabiliza el ajuste.



## MobileNetV2, MobileNetV3 y el artefacto activo

El experimento de agosto comparó MobileNetV2, EfficientNet-B0 y
MobileNetV3-Large. V3-Large ganó la validación Keras float32, pero eso no
garantiza que su TFLite sea el mejor con la cámara real.

El modelo que hoy participa en los votos es:

| Dato | Valor |
| --- | --- |
| Arquitectura | **MobileNetV2** |
| Entrada | 224 × 224 × 3, RGB crudo de 0 a 255 |
| Salidas | `plastico`, `vidrio` |
| Formato | TFLite **float32**, 8.896.164 bytes |
| Corrida | `run_20260721_2129` |
| Selección | 13 de agosto de 2026, comparación OV3660/QVGA |

Sobre 1.000 capturas QVGA, V2 obtuvo 71,60 % de exactitud y V3-Large INT8
57,10 %. Por eso V3 está en `model/backups/` y no participa en `vision_votes`.
El conjunto sirve para elegir artefactos, pero puede solaparse con desarrollo y
no sustituye una prueba reservada.



## Preprocesamiento: un detalle que importa

Cada arquitectura espera sus valores de entrada en un rango distinto:

| Arquitectura | Rango esperado |
| --- | --- |
| MobileNetV2 | de −1 a 1 |
| EfficientNet-B0 | de 0 a 255 (normaliza internamente) |
| MobileNetV3-Large | de 0 a 255 (con `include_preprocessing=True`) |

Si se alimenta un modelo con el rango equivocado, **no da error**: da
predicciones sin sentido. Es un fallo silencioso, de los peores.

Por eso los scripts de entrenamiento de Reci **hornean el preprocesamiento
dentro del modelo exportado**. El `.tflite` recibe siempre píxeles crudos de 0 a
255 y cada arquitectura resuelve su normalización por dentro. Así los tres
candidatos son intercambiables sin tocar `vision/local_model.py`.



## De dónde vino y a dónde va

El modelo activo se entrenó originalmente en RECI2 y se portó como clasificador independiente. Conserva 98,43 % de validación histórica; esa cifra no describe la precisión esperada del robot.

Con la cámara OV3660/QVGA obtuvo 71,60 % en la comparación operativa. Por eso se combina con OpenAI+sistema experto y se mantiene la salida `desconocido`. El siguiente documento explica esta brecha entre entrenamiento y producción.
